# Modeling — comparing four models and picking a champion

Part of the **Personality Profile Predictor** project: predicting a person's Big Five
personality profile from 19 questionnaire statements plus age, gender and writing hand.

This notebook picks the model that goes into the Streamlit app:

1. load the cleaned data from the EDA notebook and hold out a **test set**,
2. build the **preprocessing pipeline** that every model shares,
3. give **each of four models its own pipeline**,
4. score them all with **5-fold cross-validation** on **macro F1**,
5. **tune every model**, with grid search *and* with Hyperopt,
6. pick a **champion**, check it on the untouched test set, and save it with **joblib**,
7. document the **final decision** at the end of the notebook.

The notebook runs from top to bottom and takes about 5 minutes on a laptop.

> **Reproducibility:** every step that involves randomness uses `random_state=42`, so anyone who runs this notebook gets the same champion and the same model file.

**Where to find each part of the project brief:**

| Project checklist | Where |
|---|---|
| Comparison of at least 3 models, each with hyperparameter tuning | Sections 4–7: four models, each tuned with grid search *and* Hyperopt |
| `f1_macro` as the comparison metric | Section 3 (why), sections 5–8 (used throughout) |
| Confusion matrix / further metrics (optional) | Sections 5 and 8 |
| Saving the best model with joblib | Section 9 |
| Closing paragraph documenting the decision | Section 11 (end of the notebook) |

## Setup

All the imports and the settings to reuse later, in one place.

In [ ]:
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, Markdown, display
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             f1_score, recall_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_val_score, cross_validate, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")          # hide sklearn's convergence warnings
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42       # the same number everywhere, so results are reproducible
N_SPLITS = 5            # 5-fold cross-validation
TEST_SIZE = 0.2         # 20% of the data is held back and never used for tuning
SCORING = "f1_macro"    # our comparison metric — see section 3 for why
MAX_EVALS = 10          # how many settings Hyperopt tries per model

# StratifiedKFold keeps the class proportions the same in every fold.
# That matters here because our classes are imbalanced.
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

print("Setup ready.")

## 1. Load the data and hold out a test set

The EDA notebook cleans the raw data and saves the result as `data/data_clean.csv`. The
cleaning steps, and how many rows each one removed, are documented there.

> The `data/` folder is **not** in the repository. Download the dataset from the Drive link in the README and run the EDA notebook first, because that creates this file.

We then split off **20% as a test set** and do not touch it again until the champion is chosen. Everything in between (cross-validation, grid search, Hyperopt) happens on the training 80% only.

| Set | Used for |
|---|---|
| **Train (80%)** | Cross-validation, model comparison, hyperparameter tuning |
| **Test (20%)** | One final check of the champion on data no model has ever seen |

`stratify=y` keeps the same class proportions in both halves.

In [ ]:
df = pd.read_csv("data/data_clean.csv")

X = df.drop(columns="target")   # features: 19 answers + age + gender + hand
y = df["target"]                # what we predict, kept as readable text

# the 19 questionnaire columns = everything except the demographics and the target
item_cols = [c for c in df.columns if c not in ["age", "gender", "hand", "target"]]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,                 # same class mix in both halves
)

print("Rows total:", len(df))
print("Train:", len(X_train), "| Test:", len(X_test))
print("Features:", X.shape[1], "| questionnaire items:", len(item_cols))
df.head()

### The classes are imbalanced

This single fact decides which metric we use for the rest of the notebook.

In [ ]:
counts = y.value_counts()

target_overview = pd.DataFrame({"count": counts,"share_%": (counts / len(y) * 100).round(1)})
target_overview.index.name = "personality type"
display(target_overview)

plt.figure(figsize=(7, 3.5))
sns.countplot(x=y, order=counts.index, color="#1F6F8B")
plt.title("How many people of each personality type?")
plt.xlabel("")
plt.show()

## 2. The preprocessing pipeline

Models need numbers, with no gaps and on a comparable scale. Our columns need different
treatment:

- **numbers** (the 19 items + `age`): fill any gaps with the median, then scale
- **categories** (`gender`, `hand`): fill any gaps with the most frequent value, then one-hot encode

A `ColumnTransformer` made of two small pipelines describes this once. Every model below
reuses it, so each model sees exactly the same prepared data and the comparison is fair.

The target stays as **text** (`"Resilient"`), so predictions come out readable in the app —
no label encoder needed.

In [ ]:
# Numeric columns (the 19 items + age): fill gaps with the median, then scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical columns (gender, hand): fill gaps with the most frequent value, then one-hot encode.
# drop='first' removes one column per feature (Female and Both become the reference categories),
# because that column can always be worked out from the others.
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        # make_column_selector picks every numeric column automatically
        ('num', numeric_transformer, make_column_selector(dtype_include='number')),
        ('cat', categorical_transformer, ['gender', 'hand'])
    ]
)

preprocessor

### What the preprocessor actually does

`gender` and `hand` turn into 0/1 columns, and no gaps are left behind. We fit it on the
**training data only** here, just to look at it — inside the pipelines below it is refit
within every fold, so nothing leaks from a validation fold into the scaling.

In [ ]:
X_prepared = preprocessor.fit_transform(X_train)

print("Before:", X_train.shape, " After:", X_prepared.shape)
print("Missing values after preprocessing:", int(np.isnan(X_prepared).sum()))

feature_names = preprocessor.get_feature_names_out()
print("Last few columns:", list(feature_names[-6:])) # print the last few columns to see the one-hot encoded features for gender and hand

## 3. The metric: macro F1, not accuracy

Moderate alone is about 43% of the data. A "model" that always predicts Moderate therefore
gets 43% accuracy without learning anything — so accuracy flatters a model that ignores the
small classes.

**Macro F1** averages the F1 score of every class *equally*, so Undercontroller (12% of the
data) counts as much as Moderate. That is our comparison number from here on.

The baseline below makes the gap concrete. It is scored with the same cross-validation as
every real model.

In [ ]:
# A "model" that always predicts the most common class
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DummyClassifier(strategy='most_frequent'))
])

# Same training data and same folds as every real model below
baseline_f1 = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="f1_macro").mean()
baseline_acc = cross_val_score(baseline_pipeline, X_train, y_train, cv=cv, scoring="accuracy").mean()

print(f"Baseline (always 'Moderate'):  macro F1 = {baseline_f1:.3f}   accuracy = {baseline_acc:.3f}")
print()
print(f"So our models have to beat {baseline_f1:.2f} macro F1, not {baseline_acc:.2f} accuracy.")

## 4. Four models, four pipelines

Each model gets **its own pipeline**: the shared preprocessor in front, the classifier at the
end. They come from four different families, so the comparison tells us something about the
problem and not just about one algorithm:

| Pipeline | Model | Family | Why it is here |
|---|---|---|---|
| `pipe_logreg` | LogisticRegression | linear | Simple, fast, explainable — the one to beat |
| `pipe_knn` | KNeighborsClassifier | distance-based | Makes no assumption about the shape of the boundary |
| `pipe_rf` | RandomForestClassifier | bagged trees | Handles combinations of answers, resists overfitting |
| `pipe_hgb` | HistGradientBoostingClassifier | boosted trees | Usually the strongest on tabular data of this size |

The project brief asks for at least three models, each with tuning. We use four and tune
**all** of them.

In [ ]:
pipe_logreg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

pipe_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])

pipe_hgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])

# A dictionary so we can loop over all four below
pipelines = {
    "Logistic Regression": pipe_logreg,
    "K-Nearest Neighbors": pipe_knn,
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_hgb,
}

pipe_hgb   # show one pipeline as an example: the shared preprocessor + the classifier

## 5. Cross-validation: scoring all four fairly

A single train/test split can be lucky or unlucky. **5-fold cross-validation** trains five
times on different splits of the training data and averages the scores, which is a much
steadier estimate.

`cross_validate` gives us macro F1 *and* accuracy in one go. We also keep:

- the **standard deviation** across the folds — how stable the score is
- the **time** each model needed — part of the decision, not just the score

Every result in this notebook goes into the list `runs`, so we can build one leaderboard at the end.

In [ ]:
runs = []   # every scored model lands here, tuned or not

for name, pipe in tqdm(pipelines.items(), total=len(pipelines), desc="Cross-validating"):
    start = time.time()

    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=["f1_macro", "accuracy"])

    runs.append({
        "model": name,
        "tuning": "none",                              # not tuned yet
        "cv_f1_macro": scores["test_f1_macro"].mean(), # average over the 5 folds
        "cv_f1_std": scores["test_f1_macro"].std(),    # how much the folds disagree
        "cv_accuracy": scores["test_accuracy"].mean(),
        "test_f1_macro": np.nan,                       # only the tuned runs get a test score
        "seconds": round(time.time() - start, 1),      # how long it took
        "params": {},
        "pipeline": pipe,
    })

cv_results = pd.DataFrame(runs).sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)

cv_results[["model", "cv_f1_macro", "cv_f1_std", "cv_accuracy", "seconds"]].round(4)

### Same table as a chart, with the baseline marked

In [ ]:
plot_data = cv_results.sort_values("cv_f1_macro")   # smallest first, so the best bar is on top

plt.figure(figsize=(9, 4))
plt.barh(plot_data["model"], plot_data["cv_f1_macro"], xerr=plot_data["cv_f1_std"],
         color="#1F6F8B", height=0.6)
plt.axvline(baseline_f1, color="#C2571A", linestyle="--")
plt.text(baseline_f1 + 0.01, -0.45, "baseline (always 'Moderate')", color="#C2571A", fontsize=9)
plt.xlabel("macro F1 (5-fold cross-validation on the training set)")
plt.xlim(0, 1)
plt.title("Model comparison — before tuning")
plt.tight_layout()
plt.show()

### Where the errors are

The best model's overall score hides big differences between the classes.

`cross_val_predict` predicts every training row using a model that did **not** see that row
during training. So we get one honest prediction per person, and can read a classification
report without spending the test set.

In [ ]:
best_name = cv_results.loc[0, "model"]          # the leader before tuning
y_pred_cv = cross_val_predict(pipelines[best_name], X_train, y_train, cv=cv)

print(f"Out-of-fold predictions from {best_name}\n")
print(classification_report(y_train, y_pred_cv, digits=3))

### Confusion matrix: each row shows what share of that true class went where


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))
ConfusionMatrixDisplay.from_predictions(
    y_train, y_pred_cv,
    labels=sorted(y.unique()),
    normalize="true",   # show shares instead of counts
    cmap="Blues",
    values_format=".2f",
    colorbar=False,
    xticks_rotation=45,
    ax=ax,
)
ax.set_title(f"{best_name} — share of each true class")
plt.tight_layout()
plt.show()

**Looking at the Undercontroller row:** Its recall is far below the others, and that one class is what holds the macro F1 score down.

The reason is in the data, not the model. Undercontroller is defined by low Conscientiousness *and* low Agreeableness, but only two of the 19 statements (`C4` and `A4`) measure those two traits. Most of the signal needed to recognize it was lost when the dataset was reduced from 50 statements to 19.

Tuning cannot invent that signal — but `class_weight="balanced"` tells a model to pay more attention to the rare classes, so we put it in both searches below and let the tuning decide whether it helps.

## 6. Hyperparameter tuning — all four models

A hyperparameter is a setting we choose before training (how deep a tree may grow, how strong
the regularisation is). Tuning means searching for the combination that scores best.

The brief asks for at least three models each with tuning, so we tune **all four**, with two
methods. Both score every candidate with the same cross-validation and the same macro F1 as
section 5, so every number in this notebook stays comparable:

| Method | How it searches | Good for |
|---|---|---|
| **GridSearchCV** | Tries every combination in a list we write | Few settings, values we can name in advance |
| **Hyperopt (TPE)** | Learns from the trials it already ran and samples where the score looks promising | Many settings, or continuous ranges |

Each tuned pipeline is also scored **once** on the held-out test set. We report that number
next to the CV score, but the champion is picked by the CV score only (see section 7).

> **Note:** The brief asks for tuning on at least three models, so we tune all four. Without that rule we might have skipped Random Forest and K-Nearest Neighbors, because they were furthest behind before tuning. The results below show that would have been a mistake for Random Forest: it gains the most of all four models from tuning (about +0.04 macro F1) and ends up close to Logistic Regression.

### 6.1 Grid search

`GridSearchCV` scores **every** combination in the grid, so it is exhaustive — but the cost multiplies. Two settings with 2 and 4 values is already 8 combinations × 5 folds = 40 fits.

Note the `classifier__` prefix in the parameter names. It means *"the `C` of the pipeline step called `classifier`"*. That is how we reach inside the pipeline. Since the whole pipeline goes into the search, the preprocessor is refit inside every fold — so nothing leaks from the validation fold into the scaling.

In [ ]:
param_grids = {
    "Logistic Regression": {
        'classifier__C': [0.1, 1, 10, 100],             # regularisation strength: smaller C = simpler model
        'classifier__class_weight': [None, 'balanced'], # 'balanced' makes the rare classes count more
    },
    "K-Nearest Neighbors": {
        'classifier__n_neighbors': [11, 25, 45],        # how many neighbours vote on the class
        'classifier__weights': ['uniform', 'distance'], # 'uniform' = every neighbour counts the same,
                                                        # 'distance' = closer neighbours count more
    },
    "Random Forest": {
        'classifier__n_estimators': [200],              # number of trees (fixed, to keep the grid small)
        'classifier__max_depth': [10, 20, None],        # how deep each tree may grow (None = no limit)
        'classifier__class_weight': [None, 'balanced'],
    },
    "HistGradientBoosting": {
        'classifier__learning_rate': [0.05, 0.1],       # how much each new tree may correct the previous ones
        'classifier__max_leaf_nodes': [31, 63],         # limits tree size, which guards against overfitting
        'classifier__class_weight': [None, 'balanced'],
    },
}

for name in tqdm(list(pipelines), desc="Grid search"):
    start = time.time()

    grid_search = GridSearchCV(
        estimator=pipelines[name],      # the whole pipeline, not just the classifier
        param_grid=param_grids[name],
        cv=cv,
        scoring=SCORING,
        n_jobs=-1,                      # use all CPU cores
    )
    grid_search.fit(X_train, y_train)

    # One honest score on data the search never saw
    test_f1 = f1_score(y_test, grid_search.best_estimator_.predict(X_test), average="macro")
    before = cv_results.loc[cv_results["model"] == name, "cv_f1_macro"].iloc[0]

    runs.append({
        "model": name,
        "tuning": "grid search",
        "cv_f1_macro": grid_search.best_score_,
        "cv_f1_std": np.nan,
        "cv_accuracy": np.nan,
        "test_f1_macro": test_f1,
        "seconds": round(time.time() - start, 1),
        "params": grid_search.best_params_,
        "pipeline": grid_search.best_estimator_,
    })

    print(f"\n{name}")
    print(f"  combinations  = {len(grid_search.cv_results_['params'])}")
    print(f"  CV before     = {before:.4f}")
    print(f"  CV after      = {grid_search.best_score_:.4f}  ({grid_search.best_score_ - before:+.4f})")
    print(f"  test score    = {test_f1:.4f}")
    print(f"  best settings = {grid_search.best_params_}")
    print(f"  took          = {runs[-1]['seconds']:.0f}s")

### 6.2 Hyperopt

The same four models, but now with **continuous** ranges: `learning_rate` anywhere between
0.02 and 0.3, not just the two values the grid could afford. Hyperopt tries a setting, sees
the score, and uses that to decide what to try next — so it covers a much wider space in the
same number of fits.

Three things to know about the syntax:

- `hp.loguniform` samples across orders of magnitude (0.01, 0.1, 1, 10 equally likely)
- `scope.int(...)` is needed because `hp.quniform` returns a float, and `n_estimators` must be a whole number
- Hyperopt **minimises**, so the objective returns `-score`

In [ ]:
from hyperopt import STATUS_OK, Trials, fmin, hp, space_eval, tpe
from hyperopt.pyll.base import scope # needed to tell Hyperopt that some parameters are integers, not floats

search_spaces = {
    "Logistic Regression": {
        "C": hp.loguniform("C", np.log(0.01), np.log(100)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
    "K-Nearest Neighbors": {
        "n_neighbors": scope.int(hp.quniform("n_neighbors", 5, 60, 5)),
        "weights": hp.choice("weights", ["uniform", "distance"]),
    },
    "Random Forest": {
        "n_estimators": scope.int(hp.quniform("n_estimators", 100, 300, 50)),
        "max_depth": scope.int(hp.quniform("max_depth", 5, 25, 1)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
    "HistGradientBoosting": {
        "learning_rate": hp.loguniform("learning_rate", np.log(0.02), np.log(0.3)),
        "max_leaf_nodes": scope.int(hp.quniform("max_leaf_nodes", 15, 127, 16)),
        "min_samples_leaf": scope.int(hp.quniform("min_samples_leaf", 10, 80, 10)),
        "class_weight": hp.choice("class_weight", [None, "balanced"]),
    },
}

print("Search spaces ready for all four models.")

In [ ]:
for name in tqdm(list(pipelines), desc="Hyperopt"):
    start = time.time()
    pipe = pipelines[name]
    space = search_spaces[name]

    # Hyperopt calls this with one set of settings and wants a score back
    def objective(params):
        candidate = clone(pipe)        # a fresh copy, so nothing carries over
        candidate.set_params(**{"classifier__" + k: v for k, v in params.items()})
        score = cross_val_score(candidate, X_train, y_train, cv=cv, scoring=SCORING).mean()
        return {"loss": -score, "status": STATUS_OK}   # minus, because Hyperopt minimises

    trials = Trials()
    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,                      # the smart search strategy
        max_evals=MAX_EVALS,
        trials=trials,
        rstate=np.random.default_rng(RANDOM_STATE),
        show_progressbar=False,
    )

    # fmin returns indexes for hp.choice, so space_eval turns them back into real values
    best_params = space_eval(space, best)
    best_score = -trials.best_trial["result"]["loss"]       # flip the sign back

    # Build and fit the pipeline with those settings, then score it on the test set
    tuned_pipeline = clone(pipe)
    tuned_pipeline.set_params(**{"classifier__" + k: v for k, v in best_params.items()})
    tuned_pipeline.fit(X_train, y_train)
    test_f1 = f1_score(y_test, tuned_pipeline.predict(X_test), average="macro")

    before = cv_results.loc[cv_results["model"] == name, "cv_f1_macro"].iloc[0]

    runs.append({
        "model": name,
        "tuning": "hyperopt",
        "cv_f1_macro": best_score,
        "cv_f1_std": np.nan,
        "cv_accuracy": np.nan,
        "test_f1_macro": test_f1,
        "seconds": round(time.time() - start, 1),
        "params": {"classifier__" + k: v for k, v in best_params.items()},
        "pipeline": tuned_pipeline,
    })

    print(f"\n{name}")
    print(f"  trials        = {MAX_EVALS}")
    print(f"  CV before     = {before:.4f}")
    print(f"  CV after      = {best_score:.4f}  ({best_score - before:+.4f})")
    print(f"  test score    = {test_f1:.4f}")
    print(f"  best settings = {best_params}")
    print(f"  took          = {runs[-1]['seconds']:.0f}s")

## 7. Results — the comparison and the winner

The best tuned run of each model, shown as one card per model: the best settings the search found, how long it
took, the cross-validation score it was selected by, and its score on the held-out test set.

The **champion is the run with the highest cross-validated macro F1** — the metric we committed to in section 3. The test score is reported alongside it as an independent check, not as the selection criterion (choosing by the test score would spend the one honest number we have).

In [ ]:
all_runs = pd.DataFrame(runs)

# Only tuned runs compete for the title — every model was tuned, so none is at a disadvantage
tuned_runs = all_runs[all_runs["tuning"] != "none"].copy()
tuned_runs = tuned_runs.sort_values("cv_f1_macro", ascending=False).reset_index(drop=True)

champion = tuned_runs.loc[0]

CHAMPION_NAME = champion["model"]
CHAMPION_TUNING = champion["tuning"]
CHAMPION_CV = champion["cv_f1_macro"]
CHAMPION_TEST = champion["test_f1_macro"]
champion_pipeline = champion["pipeline"]
champion_params = champion["params"]

# Best run per model, for the cards
best_per_model = (
    tuned_runs.sort_values("cv_f1_macro", ascending=False)
    .groupby("model", as_index=False)
    .first()
    .sort_values("cv_f1_macro", ascending=False)
)

def format_params(params):
    """Turn the parameter dict into a short readable string."""
    parts = []
    for key, value in params.items():
        short = key.replace("classifier__", "") 
        if isinstance(value, float):
            value = round(value, 4)
        parts.append(f"{short}: {value}")
    return ", ".join(parts)

cards = ""
for _, row in best_per_model.iterrows():
    is_winner = row["model"] == CHAMPION_NAME
    border = "#1F6F8B" if is_winner else "#E2E0DB"
    cards += f"""
    <div style="border:2px solid {border}; border-radius:12px; padding:18px 22px;
                margin-bottom:14px; background:#FFFFFF; font-family:system-ui,sans-serif">
      <div style="display:flex; justify-content:space-between; align-items:baseline">
        <div style="font-size:19px; font-weight:600; color:#15171C">{row['model']}</div>
        <div style="font-size:13px; color:#6B7080">Duration: {row['seconds']:.1f} s
             &nbsp;·&nbsp; Tuning: {row['tuning']}</div>
      </div>
      <div style="font-size:13.5px; color:#4F5461; margin:8px 0 14px">
        <b>Best parameters:</b> {format_params(row['params'])}
      </div>
      <div style="display:flex; gap:14px">
        <div style="flex:1; background:#EAF3F7; border-radius:9px; padding:12px 16px">
          <div style="font-size:12.5px; color:#4F5461">CV score (F1 macro)</div>
          <div style="font-size:26px; font-weight:700; color:#1F6F8B">{row['cv_f1_macro']:.3f}</div>
        </div>
        <div style="flex:1; background:#EAF5EC; border-radius:9px; padding:12px 16px">
          <div style="font-size:12.5px; color:#4F5461">Test score (F1 macro)</div>
          <div style="font-size:26px; font-weight:700; color:#2E7D4F">{row['test_f1_macro']:.3f}</div>
        </div>
      </div>
    </div>"""

banner = f"""
<div style="border:2px solid #E3B341; background:#FFF9E8; border-radius:12px;
            padding:18px 22px; font-family:system-ui,sans-serif">
  <div style="font-size:19px; font-weight:700; color:#8A6400; margin-bottom:6px">
    🏆 Champion: {CHAMPION_NAME}
  </div>
  <div style="font-size:14.5px; color:#6B5200; line-height:1.6">
    Selected by <b>CV F1 macro of {CHAMPION_CV:.3f}</b> ({CHAMPION_TUNING}),
    confirmed with <b>{CHAMPION_TEST:.3f}</b> on the held-out test set.
    The baseline that always predicts "Moderate" scores {baseline_f1:.3f}.
  </div>
</div>"""

display(HTML(f"""
<h3 style="font-family:system-ui,sans-serif; margin-bottom:14px">
  Model comparison &amp; hyperparameter tuning</h3>
{cards}{banner}"""))

### The full leaderboard

Every run, including the untuned baselines, so the effect of tuning is visible.

In [ ]:
leaderboard = (
    all_runs[["model", "tuning", "cv_f1_macro", "cv_accuracy", "test_f1_macro", "seconds"]]
    .sort_values("cv_f1_macro", ascending=False)
    .reset_index(drop=True)
    .round(4)
)
display(leaderboard)

Path("results").mkdir(exist_ok=True)
leaderboard.to_csv("results/model_comparison.csv", index=False)
print("Saved results/model_comparison.csv")

## 8. The champion on the test set

The champion was chosen with cross-validation on the training data. Now we look at how it
behaves on the 20% it has never seen — per class, which is where the interesting part is.

In [ ]:
y_pred_test = champion_pipeline.predict(X_test)
champion_test_acc = accuracy_score(y_test, y_pred_test)   # used again in sections 10 and 11

print(f"{CHAMPION_NAME} ({CHAMPION_TUNING}) — held-out test set\n")
print(classification_report(y_test, y_pred_test, digits=3))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_test,
    labels=sorted(y.unique()),
    normalize="true",   # show shares instead of counts
    cmap="Greens",
    values_format=".2f",
    colorbar=False,
    xticks_rotation=45,
)
plt.title(f"Champion on the test set: {CHAMPION_NAME}")
plt.tight_layout()
plt.show()

## 9. Fit the champion on all the data and save it with joblib

Cross-validation on the training set showed which pipeline performs best, and the held-out test set confirmed that result.
Since the test score has already been measured, the test rows aren't needed for evaluation anymore. So for the final model we refit the champion pipeline on
the full dataset (training and test data combined), so it learns from every row we have before it goes into `app.py`.

We save the **complete pipeline** with `joblib`, not just the classifier. The pipeline contains
the preprocessing steps (imputation, scaling and one-hot encoding) and the tuned classifier.
When the app passes in a raw row of user answers, the pipeline transforms it exactly the same
way as the training data and then makes the prediction.

> The `models/` folder is listed in `.gitignore`, so it isn't committed to the repository.
> Running this notebook creates the model file again. Every step that involves randomness uses `random_state=42`, so you get the same model every time.

In [ ]:
Path("models").mkdir(exist_ok=True) # Create the models directory if it doesn't exist

best_pipeline = clone(champion_pipeline).fit(X, y)     # refit champion on ALL the data
joblib.dump(best_pipeline, "models/personality_pipeline.joblib")  # Save the best_pipeline final model to exact path from the brief

print(f"Saved models/personality_pipeline.joblib  ({CHAMPION_NAME}, {CHAMPION_TUNING})")
print("Classes:", list(best_pipeline.classes_))

### Check the saved file the way the app will use it

The most common reason a Streamlit app breaks is that the columns it sends don't match the
columns the pipeline was trained on. To check this, we load the saved model again and predict
from one raw row that we build by hand, the same way `app.py` does.

### Building the input dataframe

The example user answers all 19 statements (1 = Disagree, 5 = Agree) plus age, gender and writing hand. The column names come from the codebook, and they are listed in the same order as in the training data.

In [ ]:
# Load the saved pipeline again, like app.py does at startup
loaded_pipeline = joblib.load("models/personality_pipeline.joblib")

# One made-up user: a calm, sociable person
# Keys = column names from the codebook, in the same order as the training data
# Scale for the 19 statements: 1 = Disagree, 2 = Slightly Disagree, 3 = Neutral, 4 = Slightly Agree, 5 = Agree

user_answers = {
    "N6":  1,            # I get upset easily.
    "N8":  1,            # I have frequent mood swings.
    "N7":  2,            # I change my mood a lot.
    "N1":  2,            # I get stressed out easily.
    "N9":  2,            # I get irritated easily.
    "N10": 1,            # I often feel blue.
    "N3":  2,            # I worry about things.
    "N5":  1,            # I am easily disturbed.
    "E3":  5,            # I feel comfortable around people.
    "N2":  4,            # I am relaxed most of the time.
    "E5":  4,            # I start conversations.
    "N4":  4,            # I seldom feel blue.
    "E7":  4,            # I talk to a lot of different people at parties.
    "E4":  2,            # I keep in the background.
    "age": 29,           # age in years
    "C4":  2,            # I make a mess of things.
    "E1":  3,            # I am the life of the party.
    "A4":  5,            # I sympathize with others' feelings.
    "E10": 2,            # I am quiet around strangers.
    "E9":  4,            # I don't mind being the center of attention.
    "gender": "Female",  # 'Male', 'Female' or 'Other'
    "hand":   "Right",   # 'Right', 'Left' or 'Both'
}

# One row = one person
input_df = pd.DataFrame([user_answers])

# The brief's check: same column names, same order and same data types as the training data
print("Same column names and order:", list(input_df.columns) == list(X.columns))
print("Same data types:            ", input_df.dtypes.equals(X.dtypes))
input_df

### Prediction test

In [ ]:
# Predict straight from the raw row (the pipeline does all the preprocessing itself)
prediction = loaded_pipeline.predict(input_df)[0]
probabilities = loaded_pipeline.predict_proba(input_df)[0]

print("Predicted personality type:", prediction)
for personality_type, prob in zip(loaded_pipeline.classes_, probabilities):
    print(f"  {personality_type:<16} {prob:.1%}")

## 10. Results block for the README

Prints the results section to paste into `README.md`, so the champion and the metric that
chose it are recorded outside the notebook too. It is also saved as `results/README_results.md`.

In [ ]:
share = y.value_counts(normalize=True)   # class shares, for the sentence about imbalance

lines = []
lines.append("## Results")
lines.append("")
lines.append(f"**Champion model: {CHAMPION_NAME}, tuned with {CHAMPION_TUNING}**")
lines.append("")
lines.append(
    f"Selected by **macro F1** under {N_SPLITS}-fold stratified cross-validation on the training set. "
    "Macro F1 treats every personality type equally, which matters here because the classes are "
    f"imbalanced (Moderate {share['Moderate']:.0%}, Undercontroller {share['Undercontroller']:.0%})."
)
lines.append("")

# The champion's scores, compared to the baseline
lines.append(f"- Cross-validation macro F1: **{CHAMPION_CV:.3f}**")
lines.append(f"- Held-out test set macro F1: **{CHAMPION_TEST:.3f}**")
lines.append(f"- Held-out test set accuracy: **{champion_test_acc:.1%}**")
lines.append(f"- Baseline (always predicts Moderate): {baseline_f1:.3f} macro F1, "
             f"{baseline_acc:.3f} accuracy")
lines.append("")
lines.append(f"Champion settings: `{format_params(champion_params)}`")
lines.append("")

# Leaderboard table: one row per model and tuning method
lines.append("| Model | Tuning | CV F1 macro | Test F1 macro |")
lines.append("|---|---|---|---|")
for _, run in leaderboard.iterrows():
    # untuned models were never scored on the test set, so they get a dash
    test = "—" if pd.isna(run["test_f1_macro"]) else f"{run['test_f1_macro']:.3f}"
    lines.append(f"| {run['model']} | {run['tuning']} | {run['cv_f1_macro']:.3f} | {test} |")
lines.append("")
lines.append("The champion pipeline (preprocessing + classifier) was refit on the full dataset and saved "
             "as `models/personality_pipeline.joblib` for the Streamlit app.")

# Join everything, save it to results/ and print it for copy-pasting
readme_section = "\n".join(lines)
Path("results/README_results.md").write_text(readme_section, encoding="utf-8")

print(readme_section)

## 11. Final decision

The closing paragraph the project brief asks for. The numbers are filled in from the results
above, so the text always matches the latest run, even if the cleaned data changes.

In [ ]:
# ---- 1. Collect the numbers from the results above ----
untuned = all_runs[all_runs["tuning"] == "none"].sort_values("cv_f1_macro", ascending=False)
champ_runs = all_runs[(all_runs["model"] == CHAMPION_NAME) & (all_runs["tuning"] != "none")]
grid_run = champ_runs[champ_runs["tuning"] == "grid search"].iloc[0]
hyperopt_run = champ_runs[champ_runs["tuning"] == "hyperopt"].iloc[0]

untuned_champ = untuned[untuned["model"] == CHAMPION_NAME].iloc[0]
gain = CHAMPION_CV - untuned_champ["cv_f1_macro"]

# Undercontroller recall: untuned model (out-of-fold, section 5) vs. tuned champion (test set, section 8)
if best_name == CHAMPION_NAME:
    y_pred_untuned = y_pred_cv
else:
    y_pred_untuned = cross_val_predict(pipelines[CHAMPION_NAME], X_train, y_train, cv=cv)
recall_before = recall_score(y_train, y_pred_untuned, labels=["Undercontroller"], average=None)[0]
recall_after = recall_score(y_test, y_pred_test, labels=["Undercontroller"], average=None)[0]


# ---- 2. Build the two tables ----
def show(value):
    """Short readable version of a parameter value."""
    return f"{value:.3g}" if isinstance(value, float) else str(value)

model_rows = "\n".join(
    f"| **{row.model}** | **{row.cv_f1_macro:.3f}** |" if row.model == best_name
    else f"| {row.model} | {row.cv_f1_macro:.3f} |"
    for row in untuned.itertuples()
)

grid_values = param_grids[CHAMPION_NAME]
n_combinations = int(np.prod([len(v) for v in grid_values.values()]))   # size of the grid
defaults = pipelines[CHAMPION_NAME].named_steps["classifier"].get_params()
param_names = list(dict.fromkeys(list(grid_run["params"]) + list(hyperopt_run["params"])))
param_rows = []
for key in param_names:
    short = key.replace("classifier__", "")
    tried = (", ".join(show(v) for v in grid_values[key]) if key in grid_values
             else f"not searched (default {show(defaults[short])})")
    best_grid = show(grid_run["params"].get(key, defaults[short]))
    best_hyperopt = show(hyperopt_run["params"].get(key, defaults[short]))
    param_rows.append(f"| `{short}` | {tried} | {best_grid} | {best_hyperopt} |")
param_rows = "\n".join(param_rows)

if CHAMPION_TUNING == "grid search":
    winner_sentence = (f"Grid search had the higher CV score ({grid_run.cv_f1_macro:.4f} vs. "
                       f"{hyperopt_run.cv_f1_macro:.4f} for Hyperopt)")
else:
    winner_sentence = (f"Hyperopt had the higher CV score ({hyperopt_run.cv_f1_macro:.4f} vs. "
                       f"{grid_run.cv_f1_macro:.4f} for grid search)")

if champion_params.get("classifier__class_weight") == "balanced":
    class_weight_sentence = (
        f"`class_weight='balanced'` raised the recall of the smallest class, Undercontroller, from "
        f"{recall_before:.2f} (untuned, cross-validation) to {recall_after:.2f} (tuned, test set), "
        f"at the cost of a little overall accuracy ({untuned_champ.cv_accuracy:.1%} → {champion_test_acc:.1%})."
    )
else:
    class_weight_sentence = (
        f"the recall of the smallest class, Undercontroller, went from {recall_before:.2f} "
        f"(untuned, cross-validation) to {recall_after:.2f} (tuned, test set)."
    )


# ---- 3. The closing paragraph ----
final_decision = f"""
**Baseline comparison.** We first scored all four models with their default settings, using
{N_SPLITS}-fold stratified cross-validation on the training set. {best_name} showed the strongest
initial performance:

| Model | CV macro F1 (untuned) |
|---|---|
{model_rows}

**Hyperparameter tuning.** We then tuned every model twice: with GridSearchCV
({n_combinations} combinations for {CHAMPION_NAME})
and with Hyperopt ({MAX_EVALS} trials over wider, continuous ranges, see section 6.2).
For {CHAMPION_NAME}:

| Hyperparameter | Values tried in grid search | Best (grid search) | Best (Hyperopt) |
|---|---|---|---|
{param_rows}

{winner_sentence}, so the winning configuration is {format_params(champion_params)}.

**Result of the final model:**

- Cross-validated macro F1 (training set): **{CHAMPION_CV:.3f}**
- Macro F1 on the held-out 20% test set: **{CHAMPION_TEST:.3f}**
- Accuracy on the test set: **{champion_test_acc:.1%}**
- Baseline that always predicts "Moderate": {baseline_f1:.3f} macro F1

**Compared to the default parameters.** The overall gain in macro F1 is
{'small' if abs(gain) < 0.02 else 'clear'} ({gain:+.3f}, from {untuned_champ.cv_f1_macro:.3f} to {CHAMPION_CV:.3f}), but
{class_weight_sentence} Undercontroller is still the hardest class, because only two of the 19
statements (C4 and A4) measure the two traits that define it (see section 5). The
cross-validation and test scores are almost the same ({CHAMPION_CV:.3f} vs. {CHAMPION_TEST:.3f}),
so the model isn't overfitting the training data and generalises well to new data.

**Decision.** This is why we selected this pipeline (preprocessing + {CHAMPION_NAME}) as the
final model, refit it on the full dataset and saved it with joblib as
`models/personality_pipeline.joblib`. The Streamlit app (`streamlit run app.py`) loads this file
and serves the predictions.
"""

display(Markdown(final_decision))